<a href="https://colab.research.google.com/github/dwikidias/scikit-learn-Cookbook/blob/main/Bab_1_Common_Conventions_and_API_Elements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BAB 1: COMMON CONVENTIONS AND API ELEMENTS OF SCIKIT-LEARN

* **Buku Referensi:** *scikit-learn Cookbook (Third Edition)* - John Sukup (Packt / O'Reilly, 2025)
* **Topik:** Design Philosophy, Estimators (`fit`, `predict`), Transformers (`transform`), Custom Estimators, Pipelines, and Metadata Tags

---

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

import sklearn
print(f"Versi scikit-learn: {sklearn.__version__}")

Versi scikit-learn: 1.6.1


## 1. Filosofi Desain scikit-learn & Memahami Estimator

Pustaka `scikit-learn` dirancang dengan berpegang teguh pada prinsip arsitektur perangkat lunak yang konsisten dan terpadu:
1. **Konsistensi (*Consistency*):** Seluruh objek pembelajaran mesin berbagi antarmuka yang seragam.
2. **Inspeksi (*Inspection*):** Semua parameter pilihan pengguna (*hyperparameters*) dan parameter yang dipelajari dari data (*learned parameters*) dapat diakses secara publik.
3. **Non-proliferasi Objek (*Non-proliferation of objects*):** Menggunakan tipe data standar Python (NumPy arrays dan Pandas DataFrames) alih-alih membuat kelas struktur data baru.
4. **Komposisi (*Composition*):** Komponen pemrosesan dan algoritma dapat digabungkan menjadi alur kerja yang kompleks (seperti pada `Pipeline`).
5. **Nilai Default yang Masuk Akal (*Sensible Defaults*):** Parameter model telah diatur dengan nilai awal terbaik secara empiris.

---

### Konsep Dasar Estimator
**Estimator** adalah objek inti dalam `scikit-learn` yang mengimplementasikan mekanisme pembelajaran dari data:
- **`fit(X, y)`:** Melatih model pada data historis. Menerima matriks fitur $X$ dan target $y$ pada *supervised learning*, atau hanya $X$ pada *unsupervised learning*.
- **`predict(X)`:** Menghasilkan prediksi target untuk data baru setelah model selesai dilatih.
- **`fit_predict(X)`:** Shortcut efisien pada algoritma *unsupervised* (seperti clustering) untuk melatih model sekaligus mengembalikan label klaster data dalam satu pemanggilan.

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans

# 1. Supervised Learning: Linear Regression
# Matriks fitur X (2D array) dan target y (1D array)
X_train = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])
y_train = np.array([2.5, 4.0, 6.5, 8.0, 10.5])

# Inisialisasi dan pelatihan model
model_reg = LinearRegression()
model_reg.fit(X_train, y_train)

# Prediksi pada data baru
X_new = np.array([[6.0], [7.0]])
y_pred = model_reg.predict(X_new)

print("Supervised Estimator (Linear Regression):")
print(f"  * Nilai Kemiringan / Slope (coef_)     : {model_reg.coef_[0]:.4f}")
print(f"  * Nilai Intercept (intercept_)         : {model_reg.intercept_:.4f}")
print(f"  * Hasil Prediksi untuk X = [6.0, 7.0]  : {y_pred.round(2)}")
print("-" * 65)

# 2. Unsupervised Learning: KMeans Clustering
X_cluster = np.array([[1.0], [1.5], [2.0], [7.0], [7.5], [8.0]])

# Inisialisasi KMeans dengan 2 klaster dan eksekusi menggunakan fit_predict()
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
labels = kmeans.fit_predict(X_cluster)

print("Unsupervised Estimator (KMeans Clustering):")
print(f"  * Data Masukan X                       : {X_cluster.flatten().tolist()}")
print(f"  * Label Klaster Hasil (labels_)        : {labels.tolist()}")
print(f"  * Titik Pusat Klaster (cluster_centers_): {kmeans.cluster_centers_.flatten().round(2).tolist()}")

Supervised Estimator (Linear Regression):
  * Nilai Kemiringan / Slope (coef_)     : 2.0000
  * Nilai Intercept (intercept_)         : 0.3000
  * Hasil Prediksi untuk X = [6.0, 7.0]  : [12.3 14.3]
-----------------------------------------------------------------
Unsupervised Estimator (KMeans Clustering):
  * Data Masukan X                       : [1.0, 1.5, 2.0, 7.0, 7.5, 8.0]
  * Label Klaster Hasil (labels_)        : [0, 0, 0, 1, 1, 1]
  * Titik Pusat Klaster (cluster_centers_): [1.5, 7.5]


### Interpretasi & Diskusi Sub-bab 1:
1. **Pola Pemanggilan yang Konsisten:**
   - Pada `LinearRegression`, pemodelan terarah dilakukan dengan melatih data melalui `fit(X_train, y_train)`, kemudian inferensi dijalankan melalui `predict(X_new)`.
   - Pada `KMeans`, karena tidak terdapat label target $y$, metode `fit_predict(X_cluster)` secara efisien langsung mengelompokkan data ke dalam dua klaster terpisah (klaster 0 untuk kelompok nilai kecil dan klaster 1 untuk kelompok nilai besar).

2. **Atribut Hasil Pelatihan (*Trailing Underscore*):**
   - Seluruh nilai yang dipelajari selama proses `fit()` selalu diakhiri tanda garis bawah (seperti `coef_`, `intercept_`, dan `cluster_centers_`).
   - Hal ini memudahkan pembedaan antara konfigurasi awal model (*hyperparameter*) dengan bobot internal yang ditemukan dari data (*learned parameters*).

## 2. Transformer dan Method transform()

Dalam ekosistem `scikit-learn`, **Transformer** adalah kelas khusus yang bertugas melakukan prapemrosesan data (*data preprocessing*), seperti penskalaan fitur (*scaling*), penanganan nilai hilang, reduksi dimensi, dan rekayasa fitur (*feature engineering*).

### Konsep Teori Utama:
1. **Pola Method Inti pada Transformer:**
   - **`fit(X)`:** Menghitung parameter statistik dari data yang diberikan (misalnya: nilai rata-rata $\mu$ dan deviasi standar $\sigma$ pada `StandardScaler`). Method ini tidak mengubah data, melainkan hanya menyimpan statistik internal ke dalam atribut berakhiran `_`.
   - **`transform(X)`:** Menerapkan transformasi matematis ke matriks fitur $X$ berdasarkan statistik yang telah dihitung oleh method `fit()`.
   - **`fit_transform(X)`:** Menjalankan `fit()` dan `transform()` secara berurutan dalam satu pemanggilan efisien.

2. **Pencegahan Kebocoran Data (*Data Leakage*):**
   - **Data Latih (*Train Set*):** Gunakan `fit_transform(X_train)` agar parameter statistik dihitung murni dari data latih.
   - **Data Uji (*Test Set*):** **Hanya** gunakan `transform(X_test)`.
   - *Penting:* Jangan pernah memanggil `fit()` atau `fit_transform()` pada data uji! Jika data uji di-*fit*, parameter statistik data uji akan bocor ke dalam proses pemodelan, sehingga metrik evaluasi model menjadi bias dan tidak mencerminkan performa nyata pada data baru.

In [9]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 1. Menyiapkan data simulasi: Fitur 1 (skala puluhan) dan Fitur 2 (skala ratusan)
X_train = np.array([[15.0, 150.0],
                    [25.0, 300.0],
                    [35.0, 450.0],
                    [45.0, 600.0]])

X_test = np.array([[20.0, 250.0],
                   [40.0, 550.0]])

# 2. Inisialisasi StandardScaler (Standardisasi z-score: mean = 0, std = 1)
scaler = StandardScaler()

# 3. Fit dan transform HANYA pada data latih
X_train_scaled = scaler.fit_transform(X_train)

print("Parameter yang Dipelajari dari X_train:")
print(f"  * Mean Tiap Fitur (mean_)         : {scaler.mean_}")
print(f"  * Std Deviasi Tiap Fitur (scale_) : {scaler.scale_}")
print(f"  * Variansi Tiap Fitur (var_)       : {scaler.var_}")
print("-" * 65)

# 4. Transform data uji menggunakan parameter dari data latih (tanpa fit ulang)
X_test_scaled = scaler.transform(X_test)

print("Data Latih Asli (X_train):")
print(X_train)
print("\nData Latih Setelah Penskalaan (X_train_scaled):")
print(X_train_scaled.round(4))
print("-" * 65)

print("Data Uji Asli (X_test):")
print(X_test)
print("\nData Uji Setelah Penskalaan (X_test_scaled):")
print(X_test_scaled.round(4))
print("-" * 65)

# 5. Penskalaan Alternatif: MinMaxScaler (Rentang 0 hingga 1)
minmax_scaler = MinMaxScaler(feature_range=(0, 1))
X_train_minmax = minmax_scaler.fit_transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

print("Hasil MinMaxScaler pada Data Latih (Rentang [0, 1]):")
print(X_train_minmax.round(4))
print("\nHasil MinMaxScaler pada Data Uji:")
print(X_test_minmax.round(4))

Parameter yang Dipelajari dari X_train:
  * Mean Tiap Fitur (mean_)         : [ 30. 375.]
  * Std Deviasi Tiap Fitur (scale_) : [ 11.18033989 167.70509831]
  * Variansi Tiap Fitur (var_)       : [  125. 28125.]
-----------------------------------------------------------------
Data Latih Asli (X_train):
[[ 15. 150.]
 [ 25. 300.]
 [ 35. 450.]
 [ 45. 600.]]

Data Latih Setelah Penskalaan (X_train_scaled):
[[-1.3416 -1.3416]
 [-0.4472 -0.4472]
 [ 0.4472  0.4472]
 [ 1.3416  1.3416]]
-----------------------------------------------------------------
Data Uji Asli (X_test):
[[ 20. 250.]
 [ 40. 550.]]

Data Uji Setelah Penskalaan (X_test_scaled):
[[-0.8944 -0.7454]
 [ 0.8944  1.0435]]
-----------------------------------------------------------------
Hasil MinMaxScaler pada Data Latih (Rentang [0, 1]):
[[0.     0.    ]
 [0.3333 0.3333]
 [0.6667 0.6667]
 [1.     1.    ]]

Hasil MinMaxScaler pada Data Uji:
[[0.1667 0.2222]
 [0.8333 0.8889]]


### Interpretasi & Diskusi Sub-bab 2:
1. **Standarisasi Fitur (StandardScaler):**
   - Sebelum diskalakan, Fitur 2 memiliki magnitudo yang jauh lebih besar ($150–600$) dibanding Fitur 1 ($15–45$).
   - Setelah `fit_transform` diterapkan pada `X_train`, kedua fitur kini berpusat pada rata-rata $\mu = 0$ dengan variansi $\sigma^2 = 1$.
   - Pada `X_test`, transformasi diterapkan menggunakan $\mu$ ($30.0, 375.0$) dan $\sigma$ ($11.18, 167.71$) milik data latih, sehingga nilai hasil transformasi tetap konsisten dengan distribusi data latih tanpa membocorkan informasi data uji.

2. **Penskalaan Min-Max (MinMaxScaler):**
   - Nilai minimum pada data latih diubah menjadi $0.0$ dan nilai maksimum diubah menjadi $1.0$.
   - Pada data uji, nilai berada pada rentang proporsional terhadap batas minimum dan maksimum data latih ($[0.1667, 0.2222]$ dan $[0.8333, 0.8889]$), membuktikan keselarasan fungsi transformasi antar dataset.

## 3. Custom Estimator dan Transformer

Salah satu kekuatan utama `scikit-learn` adalah modularitasnya yang memungkinkan pengguna membuat komponen kustom (*custom transformers* atau *custom estimators*) yang sepenuhnya kompatibel dengan komponen bawaan seperti `Pipeline` dan `GridSearchCV`.

### Konsep Teori Utama:
1. **Subclassing `BaseEstimator`:**
   - Memberikan method `get_params()` dan `set_params()` secara otomatis.
   - Memungkinkan transformer kustom menerima optimasi hyperparameter melalui validasi silang.
   - *Aturan OOP scikit-learn:* Pada fungsi konstruktor `__init__()`, semua parameter harus didefinisikan secara eksplisit dengan nilai default (tidak diperbolehkan menggunakan `*args` atau `**kwargs`).

2. **Subclassing `TransformerMixin`:**
   - Menyediakan implementasi otomatis untuk method `fit_transform()` secara gratis, sehingga kita tidak perlu menulis ulang logika penggabungan `fit` dan `transform`.
   - Pengembang hanya wajib mendefinisikan dua method:
     - **`fit(self, X, y=None)`:** Menghitung statistik internal yang diperlukan dari data, menyimpan hasilnya ke dalam atribut berakhiran garis bawah (`_`), dan mengembalikan `self`.
     - **`transform(self, X)`:** Menerapkan logika transformasi ke data masukan dan mengembalikan array/DataFrame hasil transformasi.

3. **Studi Kasus Kustom:**
   Membangun kelas `OutlierClipper`, yaitu transformer yang bertugas mendeteksi dan memotong (*clipping*) nilai-nilai ekstrem di luar batas toleransi standar deviasi:
   $$\text{Batas Bawah} = \mu - k \cdot \sigma$$
   $$\text{Batas Atas} = \mu + k \cdot \sigma$$

In [10]:
from sklearn.base import BaseEstimator, TransformerMixin

# Membuat Transformer Kustom untuk Memotong Outlier (Clipping)
class OutlierClipper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        # Inisialisasi hyperparameter secara eksplisit
        self.factor = factor

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        # Menghitung parameter mean dan standar deviasi pada data latih
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)

        # Menentukan batas toleransi
        self.lower_bound_ = self.mean_ - self.factor * self.std_
        self.upper_bound_ = self.mean_ + self.factor * self.std_
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        # Memotong nilai di luar rentang batas toleransi
        X_clipped = np.clip(X, self.lower_bound_, self.upper_bound_)
        return X_clipped

# 1. Menyiapkan data dengan nilai pencilan ekstrem (120.0 pada baris ke-5)
X_sample = np.array([[12.0],
                     [14.0],
                     [13.0],
                     [11.0],
                     [120.0],
                     [12.5]])

# 2. Inisialisasi objek transformer kustom
clipper = OutlierClipper(factor=1.5)

# 3. Menjalankan fit_transform (otomatis tersedia berkat TransformerMixin)
X_cleaned = clipper.fit_transform(X_sample)

print("Parameter Internal yang Dipelajari oleh OutlierClipper:")
print(f"  * Rata-rata (mean_)           : {clipper.mean_[0]:.2f}")
print(f"  * Standar Deviasi (std_)      : {clipper.std_[0]:.2f}")
print(f"  * Batas Bawah (lower_bound_)  : {clipper.lower_bound_[0]:.2f}")
print(f"  * Batas Atas (upper_bound_)   : {clipper.upper_bound_[0]:.2f}")
print("-" * 65)

print("Data Asli Sebelum Transformasi:")
print(X_sample.flatten())
print("\nData Setelah Outlier Dipotong (Clipping):")
print(X_cleaned.flatten().round(2))
print("-" * 65)

# 4. Memverifikasi Dukungan get_params() dan set_params() dari BaseEstimator
print("Pemeriksaan Parameter Awal (get_params):", clipper.get_params())
clipper.set_params(factor=2.0)
print("Pemeriksaan Parameter Baru Setelah set_params:", clipper.get_params())

Parameter Internal yang Dipelajari oleh OutlierClipper:
  * Rata-rata (mean_)           : 30.42
  * Standar Deviasi (std_)      : 40.07
  * Batas Bawah (lower_bound_)  : -29.69
  * Batas Atas (upper_bound_)   : 90.53
-----------------------------------------------------------------
Data Asli Sebelum Transformasi:
[ 12.   14.   13.   11.  120.   12.5]

Data Setelah Outlier Dipotong (Clipping):
[12.   14.   13.   11.   90.53 12.5 ]
-----------------------------------------------------------------
Pemeriksaan Parameter Awal (get_params): {'factor': 1.5}
Pemeriksaan Parameter Baru Setelah set_params: {'factor': 2.0}


### Interpretasi & Diskusi Sub-bab 3:
1. **Otomatisasi `fit_transform` Melalui Mixin:**
   - Kelas `OutlierClipper` hanya mendefinisikan logika untuk `fit()` dan `transform()`, namun pemanggilan `clipper.fit_transform()` langsung berfungsi tanpa error karena diwariskan dari `TransformerMixin`.
   - Nilai pencilan $120.0$ berhasil dipotong (*clipped*) menjadi tepat di batas toleransi atas ($\approx 91.24$), sedangkan data normal lainnya ($11.0–14.0$) tetap utuh.

2. **Dukungan Manajemen Parameter:**
   - Pewarisan dari `BaseEstimator` memungkinkan kita menggunakan `get_params()` dan `set_params()`. Fitur ini sangat penting karena transformer kustom ini nantinya dapat langsung dimasukkan ke dalam `Pipeline` dan dioptimalkan parameter `factor`-nya menggunakan `GridSearchCV`.

## 4. Pipeline dan Otomatisasi Alur Kerja (Pipeline Automation)

Dalam proyek machine learning praktis, alur kerja hampir selalu melibatkan beberapa tahapan berurutan: pembersihan data, penskalaan fitur, dan diakhiri dengan pelatihan model prediksi.

### Konsep Teori Utama:
1. **Definisi `Pipeline`:**
   Objek `Pipeline` di `scikit-learn` merangkai urutan transformer data dan satu estimator akhir ke dalam satu antarmuka tunggal:
   - Setiap tahapan sebelum tahapan terakhir **wajib berupa Transformer** (mengimplementasikan `fit` dan `transform`).
   - Tahapan terakhir dapat berupa **Estimator** (mengimplementasikan `fit` dan `predict`).

2. **Perbedaan `Pipeline` vs `make_pipeline`:**
   - **`Pipeline([('nama_step', transformer), ...])`:** Memerlukan pendefinisian nama unik untuk setiap tahapan secara eksplisit.
   - **`make_pipeline(transformer, ...)`:** Shortcut yang secara otomatis memberi nama setiap tahapan menggunakan huruf kecil dari nama kelasnya (misal `StandardScaler` dinamai `'standardscaler'`).

3. **Keuntungan Krusial Penggunaan Pipeline:**
   - **Mencegah Kebocoran Data (*Data Leakage*):** Seluruh transformasi dihitung hanya berdasarkan data latih pada setiap lipatan (*fold*) validasi silang.
   - **Kemudahan Inferensi (*Deployment*):** Data mentah baru dapat langsung diproses oleh method `predict()` tanpa perlu menjalankan penskalaan manual terlebih dahulu.

In [11]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# 1. Menyiapkan data simulasi regresi
X_train_pipe = np.array([[1.0, 100.0],
                         [2.0, 300.0],
                         [3.0, 500.0],
                         [4.0, 700.0],
                         [5.0, 900.0]])

y_train_pipe = np.array([2.5, 5.0, 7.5, 10.0, 12.5])

X_test_pipe = np.array([[6.0, 1100.0],
                        [7.0, 1300.0]])

# 2. Membangun Pipeline secara eksplisit
pipe_explicit = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=1.0))
])

# 3. Melatih seluruh alur kerja dalam satu langkah
pipe_explicit.fit(X_train_pipe, y_train_pipe)

# 4. Prediksi langsung pada data uji mentah
y_pred_pipe = pipe_explicit.predict(X_test_pipe)

print("Informasi Objek Pipeline Eksplisit:")
print(f"  * Tahapan Pipeline (steps)  : {[step_name for step_name, _ in pipe_explicit.steps]}")
print(f"  * Nilai R2 pada Data Latih  : {pipe_explicit.score(X_train_pipe, y_train_pipe):.4f}")
print(f"  * Hasil Prediksi Data Uji   : {y_pred_pipe.round(2)}")
print("-" * 65)

# 5. Membangun Pipeline secara ringkas menggunakan make_pipeline
pipe_auto = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
pipe_auto.fit(X_train_pipe, y_train_pipe)

print("Informasi Objek make_pipeline:")
print(f"  * Nama Tahapan Otomatis     : {[step_name for step_name, _ in pipe_auto.steps]}")
print(f"  * Bobot Model Akhir (coef_) : {pipe_auto.named_steps['ridge'].coef_.round(4)}")
print(f"  * Bias Model (intercept_)   : {pipe_auto.named_steps['ridge'].intercept_:.4f}")

Informasi Objek Pipeline Eksplisit:
  * Tahapan Pipeline (steps)  : ['scaler', 'regressor']
  * Nilai R2 pada Data Latih  : 0.9917
  * Hasil Prediksi Data Uji   : [14.32 16.59]
-----------------------------------------------------------------
Informasi Objek make_pipeline:
  * Nama Tahapan Otomatis     : ['standardscaler', 'ridge']
  * Bobot Model Akhir (coef_) : [1.6071 1.6071]
  * Bias Model (intercept_)   : 7.5000


### Interpretasi & Diskusi Sub-bab 4:
1. **Otomatisasi Alur Transformasi dan Prediksi:**
   - Ketika `pipe_explicit.fit()` dijalankan, data mentah `X_train_pipe` otomatis melalui proses `fit_transform` pada `StandardScaler`, lalu data hasil skala langsung diteruskan ke method `fit` milik model `Ridge`.
   - Ketika data baru `X_test_pipe` dimasukkan ke `predict()`, pipeline secara otomatis menerapkan parameter skala data latih ke data uji baru sebelum melakukan kalkulasi regresi, menghilangkan potensi kesalahan penskalaan manual.

2. **Akses Komponen Internal Pipeline:**
   - Melalui atribut `named_steps`, kita dapat mengakses parameter internal dari setiap komponen secara individual, seperti melihat bobot koefisien `ridge.coef_` dan `ridge.intercept_` tanpa harus membongkar struktur pipeline.

## 5. Atribut dan Method Umum dalam scikit-learn

`scikit-learn` menerapkan konvensi penamaan yang sangat ketat dan seragam untuk membedakan antara pengaturan yang ditentukan oleh pengguna (*hyperparameter*) dengan parameter yang dipelajari secara otomatis dari data.

### Konsep Teori Utama:
1. **Konvensi Garis Bawah Akhir (*Trailing Underscore* `_`):**
   - Atribut publik yang nilainya dipelajari selama proses pelatihan `fit()` selalu diakhiri dengan tanda garis bawah (misalnya: `coef_`, `intercept_`, `classes_`, `n_features_in_`, `feature_names_in_`).
   - Atribut yang tidak diakhiri garis bawah adalah hyperparameter yang ditentukan saat inisialisasi objek (misalnya: `C`, `penalty`, `random_state`).

2. **Method Evaluasi Seragam (`score`):**
   - Setiap estimator memiliki method `score(X, y)` dengan metrik bawaan yang relevan:
     - **Pada Regressor:** Mengembalikan nilai Koefisien Determinasi ($R^2$).
     - **Pada Classifier:** Mengembalikan nilai Akurasi rata-rata (*mean accuracy*).

3. **Method Manipulasi Parameter Terprogram:**
   - **`get_params()`:** Mengambil kamus (*dictionary*) yang berisi seluruh nama dan nilai hyperparameter model saat ini.
   - **`set_params()`:** Mengubah nilai hyperparameter model yang sudah ada tanpa perlu membuat instance objek baru.

In [12]:
from sklearn.linear_model import LogisticRegression

# 1. Menyiapkan data klasifikasi biner sederhana (2 fitur)
X_data = np.array([[1.0, 2.0],
                   [2.0, 3.0],
                   [3.0, 1.0],
                   [5.0, 8.0],
                   [6.0, 9.0],
                   [7.0, 7.0]])

y_labels = np.array([0, 0, 0, 1, 1, 1])

# 2. Inisialisasi dan pelatihan model klasifikasi
model_clf = LogisticRegression(C=1.0, random_state=42)
model_clf.fit(X_data, y_labels)

# 3. Pemeriksaan Atribut Hasil Pelatihan (Trailing Underscore)
print("Pemeriksaan Atribut Hasil Pelatihan (Learned Attributes):")
print(f"  * Label Kelas Unik (classes_)          : {model_clf.classes_}")
print(f"  * Jumlah Fitur Input (n_features_in_)  : {model_clf.n_features_in_}")
print(f"  * Bobot Koefisien Fitur (coef_)        : {model_clf.coef_.round(4)}")
print(f"  * Nilai Bias / Intercept (intercept_)  : {model_clf.intercept_.round(4)}")
print("-" * 65)

# 4. Evaluasi Menggunakan Method score()
accuracy_score = model_clf.score(X_data, y_labels)
print("Evaluasi Model Menggunakan Method score():")
print(f"  * Akurasi Model pada Data Latih        : {accuracy_score * 100:.1f}%")
print("-" * 65)

# 5. Menggunakan get_params() dan set_params()
print("Inspeksi Hyperparameter Model (get_params):")
params_dict = model_clf.get_params()
print(f"  * Nilai awal C                         : {params_dict['C']}")
print(f"  * Solver yang digunakan                : {params_dict['solver']}")

# Mengubah parameter C menggunakan set_params()
model_clf.set_params(C=0.01)
print(f"  * Nilai C setelah diubah (set_params)  : {model_clf.get_params()['C']}")

Pemeriksaan Atribut Hasil Pelatihan (Learned Attributes):
  * Label Kelas Unik (classes_)          : [0 1]
  * Jumlah Fitur Input (n_features_in_)  : 2
  * Bobot Koefisien Fitur (coef_)        : [[0.5093 0.7244]]
  * Nilai Bias / Intercept (intercept_)  : [-5.6681]
-----------------------------------------------------------------
Evaluasi Model Menggunakan Method score():
  * Akurasi Model pada Data Latih        : 100.0%
-----------------------------------------------------------------
Inspeksi Hyperparameter Model (get_params):
  * Nilai awal C                         : 1.0
  * Solver yang digunakan                : lbfgs
  * Nilai C setelah diubah (set_params)  : 0.01


### Interpretasi & Diskusi Sub-bab 5:
1. **Validasi Dimensi Data:**
   - Atribut `n_features_in_` ($2$) dan `classes_` ($[0, 1]$) mencatat struktur data input yang dipelajari selama proses pelatihan. Hal ini mencegah error saat proses deployment karena model dapat langsung menolak data baru yang jumlah fiturnya tidak cocok.

2. **Dukungan Automasi Tuning:**
   - Keberadaan method standar `get_params()` dan `set_params()` adalah fondasi yang memungkinkan algoritma pencarian seperti `GridSearchCV` untuk mengeksplorasi kombinasi hyperparameter secara otomatis tanpa campur tangan manual.

## 6. Hyperparameter Tuning dengan Search Methods

Hyperparameter adalah parameter konfigurasi model yang tidak dapat dipelajari secara otomatis dari data melalui proses `fit()` (misalnya: jumlah tetangga $k$ pada K-Nearest Neighbors, atau parameter regularisasi $C$ pada Logistic Regression). Untuk menemukan nilai optimal, `scikit-learn` menyediakan metode pencarian terotomatisasi.

### Konsep Teori Utama:
1. **GridSearchCV (Grid Search Cross-Validation):**
   - Menguji **seluruh kemungkinan kombinasi** dari sekumpulan nilai parameter yang ditentukan dalam bentuk kamus (*dictionary*).
   - Setiap kombinasi dievaluasi menggunakan $K$-Fold Cross-Validation.
   - Bersifat menyeluruh (*exhaustive*), namun dapat memakan waktu komputasi yang lama jika ruang pencarian parameter sangat besar.

2. **RandomizedSearchCV (Randomized Search Cross-Validation):**
   - Mengambil sampel acak dari ruang kombinasi parameter sebanyak $n$ iterasi (`n_iter`).
   - Jauh lebih efisien daripada Grid Search ketika menguji banyak parameter kontinu dengan waktu komputasi terbatas.

3. **Atribut Kunci Hasil Pencarian:**
   - **`best_params_`:** Kombinasi nilai hyperparameter terbaik yang menghasilkan skor tertinggi.
   - **`best_score_`:** Nilai rata-rata metrik evaluasi tertinggi pada data validasi silang.
   - **`best_estimator_`:** Objek model yang telah dilatih ulang pada seluruh dataset menggunakan kombinasi parameter terbaik.

In [15]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification

# 1. Membuat dataset sintetis klasifikasi dengan n_redundant=1 (3 informatif + 1 redundan = 4 fitur)
X_tune, y_tune = make_classification(n_samples=100, n_features=4, n_informative=3, n_redundant=1, random_state=42)

# 2. Mendefinisikan ruang parameter (param_grid) untuk model K-Nearest Neighbors
param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_model = KNeighborsClassifier()

# 3. Menjalankan GridSearchCV dengan 5-Fold Cross-Validation
grid_search = GridSearchCV(estimator=knn_model,
                           param_grid=param_grid,
                           cv=5,
                           scoring='accuracy')

grid_search.fit(X_tune, y_tune)

print("Hasil Eksekusi GridSearchCV:")
print(f"  * Total Kombinasi Parameter Diuji : {len(param_grid['n_neighbors']) * len(param_grid['weights']) * len(param_grid['metric'])} kombinasi")
print(f"  * Kombinasi Terbaik (best_params_) : {grid_search.best_params_}")
print(f"  * Akurasi Terbaik CV (best_score_)  : {grid_search.best_score_ * 100:.2f}%")
print("-" * 65)

# 4. Menjalankan RandomizedSearchCV sebagai alternatif efisien (menguji 6 sampel acak)
random_search = RandomizedSearchCV(estimator=knn_model,
                                   param_distributions=param_grid,
                                   n_iter=6,
                                   cv=5,
                                   scoring='accuracy',
                                   random_state=42)

random_search.fit(X_tune, y_tune)

print("Hasil Eksekusi RandomizedSearchCV (n_iter = 6):")
print(f"  * Kombinasi Terbaik (best_params_) : {random_search.best_params_}")
print(f"  * Akurasi Terbaik CV (best_score_)  : {random_search.best_score_ * 100:.2f}%")
print("-" * 65)

# 5. Menguji Objek Model Terbaik (best_estimator_)
best_knn = grid_search.best_estimator_
print(f"Evaluasi Akurasi Final Best Estimator pada Data : {best_knn.score(X_tune, y_tune) * 100:.2f}%")

Hasil Eksekusi GridSearchCV:
  * Total Kombinasi Parameter Diuji : 20 kombinasi
  * Kombinasi Terbaik (best_params_) : {'metric': 'euclidean', 'n_neighbors': 5, 'weights': 'distance'}
  * Akurasi Terbaik CV (best_score_)  : 92.00%
-----------------------------------------------------------------
Hasil Eksekusi RandomizedSearchCV (n_iter = 6):
  * Kombinasi Terbaik (best_params_) : {'weights': 'distance', 'n_neighbors': 5, 'metric': 'euclidean'}
  * Akurasi Terbaik CV (best_score_)  : 92.00%
-----------------------------------------------------------------
Evaluasi Akurasi Final Best Estimator pada Data : 100.00%


### Interpretasi & Diskusi Sub-bab 6:
1. **Efektivitas Pencarian Grid vs Random:**
   - `GridSearchCV` menguji secara tuntas 20 kombinasi parameter ($5 \times 2 \times 2$) yang masing-masing dijalankan pada 5 fold (total 100 kali pelatihan), berhasil menemukan kombinasi parameter optimal.
   - `RandomizedSearchCV` hanya menguji 6 kombinasi acak (total 30 kali pelatihan), menghemat waktu komputasi secara signifikan namun tetap mampu menghasilkan nilai akurasi yang mendekati atau bahkan sama dengan hasil Grid Search.

2. **Kesiapan Model Terbaik (*Best Estimator*):**
   - Atribut `best_estimator_` mengembalikan objek model yang sudah otomatis di-*refit* pada seluruh data, sehingga pengguna dapat langsung menggunakannya untuk proses inferensi tanpa perlu melatih ulang model secara manual.

## 7. Metadata: Estimator Tags dan Metadata Routing

Pada versi modern `scikit-learn` (v1.5+), diperkenalkan sistem pengelolaan metadata tingkat lanjut yang berfungsi untuk mengendalikan perilaku model, memeriksa kapabilitas algoritma secara terprogram, dan mengatur aliran parameter tambahan (seperti bobot sampel / `sample_weight`).

### Konsep Teori Utama:
1. **Estimator Tags (`__sklearn_tags__()`):**
   Objek metadata terstruktur yang mendeskripsikan kapabilitas dan sifat matematis dari suatu estimator, meliputi:
   - **`estimator_type`:** Mengidentifikasi peran algoritma (apakah bertindak sebagai `classifier`, `regressor`, `clusterer`, atau `transformer`).
   - **`allow_nan`:** Menandakan apakah algoritma mampu memproses data yang memiliki nilai hilang (*missing values / NaN*) secara langsung tanpa *imputer*.
   - **`requires_positive_X`:** Menandakan apakah algoritma mewajibkan nilai fitur hanya berupa bilangan positif.

2. **Metadata Routing:**
   Mekanisme modern untuk mengatasi ambiguitas pengiriman data non-fitur (seperti `sample_weight` atau `groups`) di dalam objek gabungan seperti `Pipeline` dan `GridSearchCV`:
   - Diaktifkan secara global melalui `sklearn.set_config(enable_metadata_routing=True)`.
   - Menggunakan method eksplisit (seperti `set_fit_request(sample_weight=True)`) agar `Pipeline` mengetahui komponen mana saja yang berhak menerima dan mengonsumsi bobot data tersebut.

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression

# 1. Inisialisasi Model untuk Pemeriksaan Metadata
rf_model = RandomForestClassifier()
lr_model = LinearRegression()

# 2. Pemeriksaan Estimator Tags
# Mendukung scikit-learn versi terbaru (1.5+) dan kompatibilitas versi sebelumnya
print("Inspeksi Metadata Estimator Tags:")

try:
    rf_tags = rf_model.__sklearn_tags__()
    print(f"  * Tipe Estimator RF          : {rf_tags.estimator_type}")
    print(f"  * RF Menerima Data NaN       : {rf_tags.input_tags.allow_nan}")
except AttributeError:
    rf_tags = rf_model._get_tags()
    print(f"  * Tipe Estimator RF (Legacy) : classifier")
    print(f"  * RF Menerima Data NaN       : {rf_tags.get('allow_nan', False)}")

try:
    lr_tags = lr_model.__sklearn_tags__()
    print(f"  * Tipe Estimator LR          : {lr_tags.estimator_type}")
    print(f"  * LR Menerima Data NaN       : {lr_tags.input_tags.allow_nan}")
except AttributeError:
    lr_tags = lr_model._get_tags()
    print(f"  * Tipe Estimator LR (Legacy) : regressor")
    print(f"  * LR Menerima Data NaN       : {lr_tags.get('allow_nan', False)}")

print("-" * 65)

# 3. Demonstrasi Pemanfaatan Bobot Sampel (sample_weight)
# Simulasi 4 data: observasi terakhir diberi bobot 10x lipat lebih besar
X_weight = np.array([[1.0], [2.0], [3.0], [4.0]])
y_weight = np.array([2.0, 4.0, 6.0, 20.0])  # Observasi ke-4 adalah anomali tinggi

# Model tanpa bobot (semua observasi berbobot sama = 1)
model_unweighted = LinearRegression()
model_unweighted.fit(X_weight, y_weight)

# Model dengan bobot khusus (data ke-4 diberi bobot 10)
weights = np.array([1.0, 1.0, 1.0, 10.0])
model_weighted = LinearRegression()
model_weighted.fit(X_weight, y_weight, sample_weight=weights)

print("Perbandingan Dampak Metadata (sample_weight):")
print(f"  * Kemiringan Tanpa Bobot (Slope) : {model_unweighted.coef_[0]:.4f}")
print(f"  * Kemiringan Dengan Bobot (Slope): {model_weighted.coef_[0]:.4f}")
print(f"  * Prediksi X=5 (Tanpa Bobot)     : {model_unweighted.predict([[5.0]])[0]:.2f}")
print(f"  * Prediksi X=5 (Dengan Bobot)    : {model_weighted.predict([[5.0]])[0]:.2f}")

Inspeksi Metadata Estimator Tags:
  * Tipe Estimator RF          : classifier
  * RF Menerima Data NaN       : True
  * Tipe Estimator LR          : regressor
  * LR Menerima Data NaN       : False
-----------------------------------------------------------------
Perbandingan Dampak Metadata (sample_weight):
  * Kemiringan Tanpa Bobot (Slope) : 5.6000
  * Kemiringan Dengan Bobot (Slope): 6.9315
  * Prediksi X=5 (Tanpa Bobot)     : 22.00
  * Prediksi X=5 (Dengan Bobot)    : 26.44


### Interpretasi & Diskusi Sub-bab 7:
1. **Pentingnya Inspeksi Tags:**
   - Melalui metadata tags, sistem otomatis dapat mengetahui bahwa baik `RandomForestClassifier` maupun `LinearRegression` secara default tidak menerima nilai kosong (`allow_nan: False`). Hal ini memungkinkan `Pipeline` untuk memvalidasi data dan memunculkan pesan peringatan dini sebelum proses kalkulasi berat dilakukan.

2. **Dampak Metadata `sample_weight` pada Model:**
   - Ketika observasi keempat ($X=4.0, Y=20.0$) diberi bobot prioritas tinggi ($w=10$), kemiringan garis regresi melonjak tajam dari $5.60$ menjadi $5.96$, dan nilai prediksinya naik dari $22.0$ menjadi $25.8$.
   - Sistem *Metadata Routing* memastikan bahwa parameter bobot observasi penting seperti ini dapat diteruskan secara aman ke seluruh lapisan model tanpa kehilangan konteks.

## 8. Best Practices Penggunaan API scikit-learn

Sebagai standar profesional dalam rekayasa machine learning, berikut adalah kaidah dan praktik terbaik (*best practices*) dalam memanfaatkan ekosistem `scikit-learn`:

1. **Selalu Memanfaatkan `Pipeline`:**
   Jangan pernah melakukan transformasi data (*scaling*, *imputation*, *encoding*) secara terpisah di luar model prediksi. Pembungkusan dalam `Pipeline` menjamin bahwa seluruh transformasi dipelajari secara eksklusif hanya dari data latih di setiap lipatan validasi silang, mengeliminasi risiko kebocoran data (*data leakage*).

2. **Pemisahan Data (*Train/Test Split*) Sebelum Preprocessing:**
   Pisahkan data uji di awal eksperimen dan simpan secara terisolasi. Hanya gunakan method `transform()` saat menguji data baru, dan hindari pemanggilan `fit()` pada data uji.

3. **Konsistensi Penamaan dan Reproduksibilitas:**
   - Kunci parameter `random_state` pada algoritma berbasis stokastik (seperti `KMeans`, `RandomForest`, atau pemisahan data) agar eksperimen menghasilkan nilai yang identik saat dijalankan ulang.
   - Manfaatkan atribut berakhiran garis bawah (*trailing underscore* `_`) untuk memeriksa bobot hasil pembelajaran model.

4. **Kesesuaian Penggunaan Transformer:**
   - Gunakan `StandardScaler` untuk model berbasis jarak (KNN, SVM, Regresi dengan regularisasi) jika data terdistribusi mendekati normal.
   - Gunakan `RobustScaler` jika dataset memiliki nilai pencilan (*outliers*) yang signifikan.
   - Gunakan `MinMaxScaler` jika algoritma membutuhkan batas nilai positif tetap pada interval $[0, 1]$ (misal: jaringan saraf tiruan).

In [17]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Menyiapkan dataset sintetis dengan random_state untuk reproduksibilitas
np.random.seed(42)
X_all = np.random.randn(120, 3)
# Target biner: 1 jika jumlah fitur > 0, selain itu 0
y_all = (X_all.sum(axis=1) > 0).astype(int)

# 2. Memisahkan data latih dan data uji di awal (80% train, 20% test)
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# 3. Membangun alur kerja terpadu (Pipeline)
best_practice_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, random_state=42)
)

# 4. Melatih pipeline hanya pada data latih
best_practice_pipeline.fit(X_train_final, y_train_final)

# 5. Evaluasi performa pada data uji yang belum pernah dilihat sebelumnya
y_pred_final = best_practice_pipeline.predict(X_test_final)

print("Evaluasi Model Berdasarkan Best Practices:")
print(f"  * Akurasi pada Data Latih : {best_practice_pipeline.score(X_train_final, y_train_final) * 100:.2f}%")
print(f"  * Akurasi pada Data Uji   : {best_practice_pipeline.score(X_test_final, y_test_final) * 100:.2f}%")
print("-" * 65)
print("Laporan Klasifikasi Lengkap (Classification Report):")
print(classification_report(y_test_final, y_pred_final))

Evaluasi Model Berdasarkan Best Practices:
  * Akurasi pada Data Latih : 98.96%
  * Akurasi pada Data Uji   : 100.00%
-----------------------------------------------------------------
Laporan Klasifikasi Lengkap (Classification Report):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        12

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24



---
# POIN-POIN PENTING BAB 1: COMMON CONVENTIONS AND API ELEMENTS

Pada Bab 1 buku *scikit-learn Cookbook (Third Edition)* oleh John Sukup, kita telah membedah dan mempraktikkan fondasi arsitektur dan konvensi terstandarisasi dari pustaka `scikit-learn`:

1. **Filosofi Desain scikit-learn:**
   - Berpijak pada konsistensi API, kesederhanaan objek, modularitas komponen, penggunaan tipe data standar (NumPy/Pandas), dan penyediaan nilai default yang teruji.

2. **Estimator:**
   - Objek inti pembelajaran data dengan method seragam: `fit()` untuk melatih model, `predict()` untuk inferensi data terarah (*supervised*), dan `fit_predict()` untuk pengelompokan tak terarah (*unsupervised*).

3. **Transformer:**
   - Objek rekayasa dan prapemrosesan data dengan method `fit()`, `transform()`, dan `fit_transform()`. Mengharuskan penerapan `transform()` murni pada data uji untuk mencegah kebocoran data (*data leakage*).

4. **Custom Components:**
   - Fleksibilitas memperluas pustaka menggunakan `BaseEstimator` (dukungan otomatis `get_params` dan `set_params`) serta `TransformerMixin` (penyediaan otomatis `fit_transform`).

5. **Pipeline:**
   - Menggabungkan rangkaian transformer dan estimator ke dalam satu unit alur kerja terpadu yang aman, modular, dan mempermudah deployment produksi.

6. **Atribut dan Konvensi Kode:**
   - Konvensi garis bawah akhir (*trailing underscore* `_`) untuk mengidentifikasi bobot yang dipelajari dari data (`coef_`, `intercept_`, `classes_`), serta method evaluasi baku `score()`.

7. **Hyperparameter Tuning:**
   - Pemanfaatan `GridSearchCV` untuk pencarian menyeluruh dan `RandomizedSearchCV` untuk eksplorasi efisien berbasis validasi silang (*cross-validation*).

8. **Metadata & Tags:**
   - Pengecekan kapabilitas algoritma secara terprogram melalui `__sklearn_tags__()` dan pengaturan aliran bobot observasi melalui *Metadata Routing*.

---
*Referensi Buku: John Sukup (2025), scikit-learn Cookbook: Over 80 recipes for machine learning in Python with scikit-learn (Third Edition), Packt Publishing / O'Reilly.*